In [1]:
import tensorflow as tf
import numpy as np
import cv2
import streamlit

ModuleNotFoundError: No module named 'pillow'

In [6]:
import os

print(os.getcwd())

C:\Users\ferna


In [2]:
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

TensorFlow: 2.21.0
Keras: 3.15.0


In [6]:
import os

ruta = r"C:\Users\ferna\PROYECTO IA\converted_savedmodel"

print(os.listdir(ruta))

['labels.txt', 'model.savedmodel']


In [7]:
import os

ruta = r"C:\Users\ferna\PROYECTO IA\converted_savedmodel\model.savedmodel"

print(os.listdir(ruta))

['assets', 'saved_model.pb', 'variables']


In [9]:
import tensorflow as tf

modelo = tf.saved_model.load(r"C:\Users\ferna\PROYECTO IA\converted_savedmodel\model.savedmodel")

In [10]:
with open(r"C:\Users\ferna\PROYECTO IA\converted_savedmodel\labels.txt", "r") as f:
    labels = [line.strip() for line in f.readlines()]

print(labels)

['0 Papel o Carton', '1 Vidrios', '2 PlÃ¡stico']


In [11]:
import cv2

print(cv2.__version__)

5.0.0


In [18]:
import cv2

ruta = r"C:\Users\ferna\PROYECTO IA\San luis.jpg.jpg"

imagen = cv2.imread(ruta)

if imagen is None:
    print("No se pudo cargar la imagen.")
else:
    print("Imagen cargada correctamente.")

Imagen cargada correctamente.


In [19]:
import numpy as np
import cv2

# Cambiar tamaño a 224x224 (tamaño usado por Teachable Machine)
imagen = cv2.resize(imagen, (224, 224))

# Convertir de BGR a RGB
imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)

# Convertir a float32
imagen = imagen.astype(np.float32)

# Normalizar entre -1 y 1
imagen = (imagen / 127.5) - 1

# Agregar dimensión del lote (batch)
imagen = np.expand_dims(imagen, axis=0)

print(imagen.shape)

(1, 224, 224, 3)


In [20]:
print(modelo.signatures)

_SignatureMap({'serving_default': <ConcreteFunction (*, sequential_5_input: TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='sequential_5_input')) -> Dict[['sequential_7', TensorSpec(shape=(None, 3), dtype=tf.float32, name='sequential_7')]] at 0x1F8A6B3FE30>})


In [21]:
infer = modelo.signatures["serving_default"]

In [22]:
import tensorflow as tf
import numpy as np

# Convertir la imagen a tensor
tensor = tf.convert_to_tensor(imagen)

# Ejecutar el modelo
resultado = infer(sequential_5_input=tensor)

print(resultado)

{'sequential_7': <tf.Tensor: shape=(1, 3), dtype=float32, numpy=array([[0.00066086, 0.01484428, 0.98449486]], dtype=float32)>}


In [23]:
predicciones = resultado["sequential_7"].numpy()

print(predicciones)

[[0.00066086 0.01484428 0.98449486]]


In [24]:
indice = np.argmax(predicciones)

print("Clase detectada:", labels[indice])
print("Confianza:", predicciones[0][indice] * 100, "%")

Clase detectada: 2 PlÃ¡stico
Confianza: 98.449486 %


In [25]:
import cv2
import tensorflow as tf
import numpy as np

cap = cv2.VideoCapture(0)

infer = modelo.signatures["serving_default"]

while True:
    ret, frame = cap.read()

    if not ret:
        break

    # Preparar imagen
    img = cv2.resize(frame, (224, 224))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32)
    img = (img / 127.5) - 1
    img = np.expand_dims(img, axis=0)

    tensor = tf.convert_to_tensor(img)

    resultado = infer(sequential_5_input=tensor)
    predicciones = resultado["sequential_7"].numpy()

    indice = np.argmax(predicciones)
    confianza = predicciones[0][indice] * 100
    clase = labels[indice]

    cv2.putText(frame,
                f"{clase} ({confianza:.1f}%)",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 0),
                2)

    cv2.imshow("Clasificador de Reciclaje", frame)

    if cv2.waitKey(1) & 0xFF == 27:   # ESC para salir
        break

cap.release()
cv2.destroyAllWindows()